In [1]:
!pip install pandas numpy sentence-transformers faiss-cpu openai torch

In [2]:
import pandas as pd
import numpy as np
from typing import List, Dict
import json
import os
from sentence_transformers import SentenceTransformer
import faiss
from openai import OpenAI

class MoviePlotRAG:
    def __init__(self, openai_api_key: str = None):
        """Initialize the RAG system with embeddings model and vector store."""
        print("Initializing RAG system...")

        # Initialize embedding model (using sentence-transformers for simplicity)
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.embedding_dim = 384  # Dimension for all-MiniLM-L6-v2

        # Initialize FAISS index
        self.index = None
        self.chunks = []
        self.chunk_metadata = []

        # Initialize OpenAI client (optional - can be replaced with other LLMs)
        self.client = OpenAI(api_key=openai_api_key) if openai_api_key else None

        print("✓ RAG system initialized")

    def load_and_preprocess_data(self, csv_path: str, num_movies: int = 300):
        """Load and preprocess movie plot data."""
        print(f"\nLoading data from {csv_path}...")

        # Load the dataset
        df = pd.read_csv(csv_path, engine='python', on_bad_lines='skip') # Added on_bad_lines='skip'

        # Select subset and clean
        df = df[['Title', 'Plot']].dropna()
        df = df.head(num_movies)

        print(f"✓ Loaded {len(df)} movies")
        return df

    def chunk_text(self, text: str, chunk_size: int = 300) -> List[str]:
        """Split text into chunks of approximately chunk_size words."""
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size):
            chunk = ' '.join(words[i:i + chunk_size])
            chunks.append(chunk)

        return chunks

    def build_vector_store(self, df: pd.DataFrame):
        """Chunk plots, create embeddings, and build FAISS index."""
        print("\nBuilding vector store...")

        all_embeddings = []

        for idx, row in df.iterrows():
            title = row['Title']
            plot = row['Plot']

            # Chunk the plot
            chunks = self.chunk_text(plot)

            for chunk in chunks:
                self.chunks.append(chunk)
                self.chunk_metadata.append({
                    'title': title,
                    'chunk': chunk
                })

        print(f"Created {len(self.chunks)} chunks from {len(df)} movies")

        # Create embeddings for all chunks
        print("Generating embeddings...")
        all_embeddings = self.embedding_model.encode(
            self.chunks,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # Build FAISS index
        print("Building FAISS index...")
        self.index = faiss.IndexFlatL2(self.embedding_dim)
        self.index.add(all_embeddings.astype('float32'))

        print(f"✓ Vector store built with {self.index.ntotal} vectors")

    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """Retrieve top-k most relevant chunks for a query."""
        # Embed the query
        query_embedding = self.embedding_model.encode([query], convert_to_numpy=True)

        # Search in FAISS
        distances, indices = self.index.search(query_embedding.astype('float32'), top_k)

        # Gather results
        results = []
        for i, idx in enumerate(indices[0]):
            results.append({
                'title': self.chunk_metadata[idx]['title'],
                'chunk': self.chunk_metadata[idx]['chunk'],
                'distance': float(distances[0][i])
            })

        return results

    def generate_answer(self, query: str, contexts: List[Dict]) -> Dict:
        """Generate answer using LLM with retrieved contexts."""

        # Prepare context string
        context_str = "\n\n".join([
            f"Movie: {ctx['title']}\nPlot excerpt: {ctx['chunk']}"
            for ctx in contexts
        ])

        # Create prompt
        prompt = f"""Based on the following movie plot excerpts, answer the question.\n\nMovie Plot Contexts:\n{context_str}\n\nQuestion: {query}\n\nProvide a clear, concise answer based on the information in the contexts. If the information is not in the contexts, say so."""

        if self.client:
            # Use OpenAI API
            try:
                response = self.client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant that answers questions about movies based on provided plot information."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.7,
                    max_tokens=300
                )
                answer = response.choices[0].message.content
            except Exception as e:
                answer = f"Error generating answer: {str(e)}"
        else:
            # Fallback: Simple extraction-based answer
            answer = self._generate_simple_answer(query, contexts)

        # Generate reasoning
        reasoning = f"Searched through {len(self.chunks)} plot chunks. Retrieved top {len(contexts)} most relevant contexts from movies: {', '.join(set(ctx['title'] for ctx in contexts))}. Used these contexts to form an answer."

        return {
            "answer": answer,
            "contexts": [f"{ctx['title']}: {ctx['chunk'][:200]}..." for ctx in contexts],
            "reasoning": reasoning
        }

    def _generate_simple_answer(self, query: str, contexts: List[Dict]) -> str:
        """Generate a simple answer without LLM API (fallback method)."""
        # This is a simple fallback - just return the most relevant context
        if contexts:
            best_context = contexts[0]
            return f"Based on the movie '{best_context['title']}': {best_context['chunk'][:300]}..."
        return "No relevant information found in the movie plots."

    def query(self, question: str, top_k: int = 3) -> Dict:
        """Main query method: retrieve contexts and generate answer."""
        print(f"\nQuery: {question}")
        print("Retrieving relevant contexts...")

        # Retrieve relevant chunks
        contexts = self.retrieve(question, top_k=top_k)

        print(f"✓ Retrieved {len(contexts)} contexts")
        print("Generating answer...")

        # Generate answer
        result = self.generate_answer(question, contexts)

        print("✓ Answer generated")
        return result


def main():
    """Main execution function."""
    print("=" * 60)
    print("MOVIE PLOT RAG SYSTEM")
    print("=" * 60)

    # Configuration
    CSV_PATH = "wiki_movie_plots_deduped.csv"
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

    # Initialize RAG system
    rag = MoviePlotRAG(openai_api_key=OPENAI_API_KEY)

    # Load and preprocess data
    df = rag.load_and_preprocess_data(CSV_PATH, num_movies=300)

    # Build vector store
    rag.build_vector_store(df)

    # Example queries
    queries = [
        "What movie features an artificial intelligence system?",
        "Tell me about a movie with time travel",
        "Which movie has a character named HAL 9000?"
    ]

    results = []

    for query in queries:
        result = rag.query(query, top_k=3)
        results.append({
            "query": query,
            "result": result
        })

        # Pretty print
        print("\n" + "=" * 60)
        print(f"QUERY: {query}")
        print("-" * 60)
        print(f"ANSWER: {result['answer']}")
        print(f"\nREASONING: {result['reasoning']}")
        print(f"\nCONTEXTS USED:")
        for i, ctx in enumerate(result['contexts'], 1):
            print(f"  {i}. {ctx}")
        print("=" * 60)

    # Save results to JSON
    output_file = "rag_results.json"
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\n✓ Results saved to {output_file}")

    # Interactive mode (optional)
    print("\n" + "=" * 60)
    print("INTERACTIVE MODE (type 'quit' to exit)")
    print("=" * 60)

    while True:
        user_query = input("\nYour question: ").strip()
        if user_query.lower() in ['quit', 'exit', 'q']:
            break

        if user_query:
            result = rag.query(user_query, top_k=3)
            print(f"\n{result['answer']}")


if __name__ == "__main__":
    main()

MOVIE PLOT RAG SYSTEM
Initializing RAG system...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


✓ RAG system initialized

Loading data from wiki_movie_plots_deduped.csv...
✓ Loaded 300 movies

Building vector store...
Created 353 chunks from 300 movies
Generating embeddings...


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Building FAISS index...
✓ Vector store built with 353 vectors

Query: What movie features an artificial intelligence system?
Retrieving relevant contexts...
✓ Retrieved 3 contexts
Generating answer...
✓ Answer generated

QUERY: What movie features an artificial intelligence system?
------------------------------------------------------------
ANSWER: Based on the movie 'Youth's Endearing Charm': The film is about a court case and embezzlement....

REASONING: Searched through 353 plot chunks. Retrieved top 3 most relevant contexts from movies: The Immigrant, Youth's Endearing Charm, One A.M.. Used these contexts to form an answer.

CONTEXTS USED:
  1. Youth's Endearing Charm: The film is about a court case and embezzlement....
  2. The Immigrant: The film begins aboard a steamer crossing the Atlantic Ocean, and initially showcases the misadventures of an unnamed immigrant, the Tramp (Chaplin) who finds himself in assorted mischief while, among...
  3. One A.M.: The film opens with a scen